In [ ]:
import nltk
nltk.download("punkt")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/flickr_dataset /content/


In [ ]:
import os

CAPTION_FILE = "/content/flickr_dataset/Flickr8k.token.txt"
def load_captions(caption_file):
    captions = []
    image_names = []

    with open(caption_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if len(line) == 0:
                continue

            # Split by tab (Flickr8k uses tab separator)
            parts = line.split("\t")
            if len(parts) != 2:
                continue  # Skip malformed lines

            image_caption_id, caption = parts
            # Extract just the image name (before #)
            image_with_id = image_caption_id.split("#")[0]

            # Remove any trailing numbers or dots after .jpg
            if image_with_id.endswith('.jpg'):
                image = image_with_id
            else:
                # Handle cases like "image.jpg.1" -> "image.jpg"
                image = image_with_id.rsplit('.', 1)[0] if '.' in image_with_id else image_with_id
                if not image.endswith('.jpg'):
                    image = image + '.jpg'

            image_names.append(image)
            captions.append(caption)

    return image_names, captions


In [ ]:
import re

def clean_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"[^a-z ]", "", caption)  # remove punctuation & numbers
    caption = re.sub(r"\s+", " ", caption)    # remove extra spaces
    caption = caption.strip()

    caption = "<start> " + caption + " <end>"
    return caption


In [ ]:
def preprocess_captions(captions):
    cleaned = []
    for cap in captions:
        cleaned.append(clean_caption(cap))
    return cleaned


In [ ]:
from collections import Counter
def build_vocabulary(captions, freq_threshold=5):
    counter = Counter()

    for caption in captions:
        counter.update(caption.split())
    vocab = [word for word in counter if counter[word] >= freq_threshold]


    return vocab, counter

In [ ]:
def create_word_mappings(vocab):
    word2idx = {
        "<pad>": 0,
        "<start>": 1,
        "<end>": 2
    }

    idx2word = {
        0: "<pad>",
        1: "<start>",
        2: "<end>"
    }

    idx = 3
    for word in vocab:
        if word not in word2idx:
            word2idx[word] = idx
            idx2word[idx] = word
            idx += 1

    return word2idx, idx2word


In [ ]:
def numericalize_captions(captions, word2idx):
    numeric_captions = []

    for caption in captions:
        tokens = caption.split()
        numeric = []

        for word in tokens:
            if word in word2idx:
                numeric.append(word2idx[word])
            else:
                numeric.append(word2idx["<pad>"])

        numeric_captions.append(numeric)

    return numeric_captions


In [ ]:
def main():
    image_names, captions = load_captions(CAPTION_FILE)

    captions = preprocess_captions(captions)

    vocab, counter = build_vocabulary(captions, freq_threshold=5)

    word2idx, idx2word = create_word_mappings(vocab)

    numeric_captions = numericalize_captions(captions, word2idx)

    print("Total images:", len(set(image_names)))
    print("Total captions:", len(captions))
    print("Vocabulary size:", len(word2idx))

    print("\nExample:")
    print("Caption:", captions[0])
    print("Numerical:", numeric_captions[0])
    print("Most common words:")

    for word, count in counter.most_common(10):
       print(word, count)


if __name__ == "__main__":
    main()


Total images: 8092
Total captions: 40460
Vocabulary size: 2987

Example:
Caption: <start> a child in a pink dress is climbing up a set of stairs in an entry way <end>
Numerical: [1, 3, 4, 5, 3, 6, 7, 8, 9, 10, 3, 11, 12, 13, 5, 14, 0, 15, 2]
Most common words:
a 62989
<start> 40460
<end> 40460
in 18975
the 18419
on 10744
is 9345
and 8852
dog 8136
with 7765


In [ ]:
#no pading here now as padding depedns on batch size(number of samples used in one forward + backward pass) during training
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

class FlickrDataset(Dataset):
    def __init__(self, image_dir, image_names, captions, transform):
        self.image_dir = image_dir
        self.image_names = image_names
        self.captions = captions
        self.transform = transform

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_names[idx])
        try:
            image = Image.open(image_path).convert("RGB")
            if self.transform:
              image = self.transform(image)

              caption = torch.tensor(self.captions[idx])

            return image, caption
        except FileNotFoundError:
          return None



In [ ]:
import torch
torch.cuda.is_available()


False

In [ ]:
def collate_fn(batch):
     # Remove samples that failed to load (None)
    batch = [item for item in batch if item is not None]

    # If all samples in this batch are invalid, skip it
    if len(batch) == 0:
        return None
    batch.sort(key=lambda x: len(x[1]), reverse=True)#sort by caption length

    images, captions = zip(*batch) #separate images and captions

    images = torch.stack(images, dim=0) #stack images into a single tensor , now images is of shape (batch_size, 3, 224, 224)

    lengths = [len(caption) for caption in captions]
    max_len = max(lengths)

    padded_captions = torch.zeros(len(captions), max_len).long() #padding done here

    for i, caption in enumerate(captions):
        end = lengths[i]
        padded_captions[i, :end] = caption

    return images, padded_captions, lengths


In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
image_names, captions = load_captions(CAPTION_FILE)
captions = preprocess_captions(captions)

vocab, counter = build_vocabulary(captions, freq_threshold=5)

word2idx, idx2word = create_word_mappings(vocab)

numeric_captions = numericalize_captions(captions, word2idx)

dataset = FlickrDataset(
    image_dir="/content/flickr_dataset/Flicker8k_Dataset",
    image_names=image_names,
    captions=numeric_captions,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,      # IMPORTANT
    pin_memory=True
)


In [ ]:
images, captions, lengths = next(iter(loader))

print("Images shape:", images.shape)# 32 is batch size, 3 colour channels, 224 = height, 224 = width
print("Captions shape:", captions.shape) # (batch_size, max_caption_length_in_batch)
print("Lengths:", lengths[:5]) # lengths of the captions in the batch


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Images shape: torch.Size([32, 3, 224, 224])
Captions shape: torch.Size([32, 23])
Lengths: [23, 23, 23, 21, 21]


In [ ]:
#cnn encoding
#converting images to numeric feature vectors instead of  normal words i.e not image=dog but image = numerical feature vector and then fed into lstm
#we use pre trained cnn like resnet to extract features from images and then feed those features to lstm for caption generation
'''Training a CNN from scratch needs:
Millions of images
Huge compute
So we use transfer learning:
CNN already learned edges, shapes, objects
We reuse that knowledge
For Flickr8k:
 ResNet-50 is perfect'''

'Training a CNN from scratch needs:\nMillions of images\nHuge compute\nSo we use transfer learning:\nCNN already learned edges, shapes, objects\nWe reuse that knowledge\nFor Flickr8k:\n ResNet-50 is perfect'

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EncoderCNN(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(pretrained=True)

        # Remove avgpool + fc
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])

        for param in self.resnet.parameters():
            param.requires_grad = False

    def forward(self, images):
        features = self.resnet(images)
        # (B, 2048, 7, 7)

        features = features.permute(0, 2, 3, 1)
        # (B, 7, 7, 2048)

        features = features.view(features.size(0), -1, 2048)
        # (B, 49, 2048)

        return features



In [ ]:
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, att_dim):
        super().__init__()
        self.enc = nn.Linear(encoder_dim, att_dim)
        self.dec = nn.Linear(decoder_dim, att_dim)
        self.fc = nn.Linear(att_dim, 1)

    def forward(self, encoder_out, hidden):
        att = self.fc(torch.tanh(
            self.enc(encoder_out) + self.dec(hidden).unsqueeze(1)
        )).squeeze(2)

        alpha = torch.softmax(att, dim=1)
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)
        return context


In [ ]:
#testing encoder
encoder = EncoderCNN()

images, captions, lengths = next(iter(loader))

features = encoder(images)

print("Image batch:", images.shape)
print("Feature batch:", features.shape)


Image batch: torch.Size([32, 3, 224, 224])
Feature batch: torch.Size([32, 49, 2048])


In [ ]:
#these features vector can be fed into decoder lstm for caption/setence generation
#decoder lstm will do :-Image features + previous words → predict next word
'''
CNN gives a fixed image vector
LSTM reads:
   1)Image feature (only once, at the start)
   2)Caption words (during training)
LSTM predicts the next word at each timestep
'''


'\nCNN gives a fixed image vector\nLSTM reads:\n   1)Image feature (only once, at the start)\n   2)Caption words (during training)\nLSTM predicts the next word at each timestep\n'

In [ ]:
import torch
import torch.nn as nn

class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size):
        super().__init__()

        self.hidden_size = hidden_size  # ✅ STORE IT

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.attention = Attention(2048, hidden_size, 512)
        self.lstm = nn.LSTMCell(embed_size + 2048, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, encoder_out, captions):
        batch_size = encoder_out.size(0)

        # ✅ USE self.hidden_size
        hidden = torch.zeros(batch_size, self.hidden_size).to(encoder_out.device)
        cell   = torch.zeros(batch_size, self.hidden_size).to(encoder_out.device)

        embeddings = self.embedding(captions[:, :-1])
        outputs = []

        for t in range(embeddings.size(1)):
            context = self.attention(encoder_out, hidden)
            lstm_input = torch.cat([embeddings[:, t], context], dim=1)
            hidden, cell = self.lstm(lstm_input, (hidden, cell))
            outputs.append(self.fc(hidden))

        return torch.stack(outputs, dim=1)




In [ ]:
def beam_search(encoder_out, decoder, word2idx, idx2word, beam_size=3, max_len=20):

    k = beam_size
    vocab_size = len(word2idx)

    encoder_out = encoder_out.expand(k, encoder_out.size(1), encoder_out.size(2))

    sequences = [[[], 0.0]]
    hidden = torch.zeros(k, hidden_size).to(encoder_out.device)
    cell = torch.zeros(k, hidden_size).to(encoder_out.device)

    for _ in range(max_len):
        all_candidates = []

        for i, (seq, score) in enumerate(sequences):
            if len(seq) > 0 and seq[-1] == word2idx["<end>"]:
                all_candidates.append((seq, score))
                continue

            word = torch.tensor([seq[-1]] if seq else [word2idx["<start>"]]).to(encoder_out.device)
            embed = decoder.embedding(word)

            context, _ = decoder.attention(encoder_out[i:i+1], hidden[i:i+1])
            lstm_input = torch.cat([embed, context], dim=1)

            h, c = decoder.lstm(lstm_input, (hidden[i:i+1], cell[i:i+1]))
            scores = decoder.fc(h)

            topk = scores.topk(k)

            for j in range(k):
                candidate = (seq + [topk.indices[0][j].item()],
                             score - topk.values[0][j].item())
                all_candidates.append(candidate)

        sequences = sorted(all_candidates, key=lambda x: x[1])[:k]

    best_seq = sequences[0][0]
    caption = [idx2word[idx] for idx in best_seq if idx2word[idx] not in ["<start>", "<end>"]]

    return " ".join(caption)


In [ ]:
'''
Assume:
batch_size = 32
embed_size = 512
hidden_size = 512
max caption length = 17

features        → (32, 512)
captions        → (32, 17)

captions[:, :-1]→ (32, 16)
embeddings      → (32, 16, 512)

concat features → (32, 17, 512)

LSTM output     → (32, 17, 512)
FC output       → (32, 17, vocab_size)

we input the image embedding from the encoder AND the caption for that image into the LSTM, and train the LSTM to predict the next word.

or one training example:

Given:

🖼️ Image → Encoder → image feature vector

📝 Caption → numerical tokens

Example caption:

<start> a dog is running <end>


Numerical:

[1, 45, 92, 18, 203, 2]

🔹 Inputs to the LSTM
Step-by-step input sequence
Time step	Input to LSTM
t = 0	Image embedding
t = 1	<start>
t = 2	a
t = 3	dog
t = 4	is
t = 5	running

So the LSTM input sequence is:

we give image only once at time step 0 s image does not change in time and lstm remembers it in hidden state
[IMAGE, <start>, a, dog, is, running]


This is exactly what this line does

embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)

🔹 Targets (what the LSTM must predict)

The target sequence is the caption shifted by one word:

Time step	Target word
t = 0	<start>
t = 1	a
t = 2	dog
t = 3	is
t = 4	running
t = 5	<end>


doing teacher forcing here i.e we give the correct previous word as input to predict next word during training
'''


In [ ]:
#testing
encoder = EncoderCNN()
decoder = DecoderRNN(
    embed_size=512,
    hidden_size=512,
    vocab_size=len(word2idx)
)

features = encoder(images)
print("Encoder output:", features.shape)
# (B, 49, 2048)

outputs = decoder(features, captions)
print("Decoder output:", outputs.shape)
# (B, seq_len-1, vocab_size)


Encoder output: torch.Size([32, 49, 2048])
Decoder output: torch.Size([32, 19, 2987])


In [ ]:
'''
TRAINING LOOP (CNN + LSTM)
🔹 8.0 What training really means here

For each batch:

Encode images → image embeddings

Feed embeddings + captions to LSTM

Predict next word at every timestep

Compare with ground truth

Backpropagate error

Update weights

🔹 8.1 What is INPUT and TARGET (CRUCIAL)

Example caption:

<start> a dog is running <end>

Model INPUT:
[IMAGE, <start>, a, dog, is, running]

Model TARGET:
[<start>, a, dog, is, running, <end>]


This is why we:

captions[:, :-1]   # input
captions[:, 1:]    # target
'''

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# HYPERPARAMETERS
embed_size = 512
hidden_size = 512
batch_size = 32
epochs = 5
lr = 0.001
vocab_size = len(word2idx)

encoder = EncoderCNN().to(device)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(decoder.parameters(), lr=lr)

for epoch in range(epochs):
    encoder.train()
    decoder.train()
    total_loss = 0

    for images, captions, _ in tqdm(loader):
        images, captions = images.to(device), captions.to(device)

        optimizer.zero_grad()
        features = encoder(images)
        outputs = decoder(features, captions)

        targets = captions[:, 1:]
        loss = criterion(
            outputs.reshape(-1, vocab_size),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}  Loss: {total_loss/len(loader):.4f}")


 23%|██▎       | 287/1265 [45:25<2:34:49,  9.50s/it]


KeyboardInterrupt: 

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


False
NO GPU


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/image_captioning"
import os
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
torch.save(encoder.state_dict(), f"{SAVE_DIR}/encoder.pth")
torch.save(decoder.state_dict(), f"{SAVE_DIR}/decoder.pth")

print("Models saved to Google Drive")


In [ ]:
import pickle

with open(f"{SAVE_DIR}/word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)

with open(f"{SAVE_DIR}/idx2word.pkl", "wb") as f:
    pickle.dump(idx2word, f)

print("Vocabulary saved")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pickle

with open("/content/drive/MyDrive/image_captioning/word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)

with open("/content/drive/MyDrive/image_captioning/idx2word.pkl", "rb") as f:
    idx2word = pickle.load(f)

vocab_size = len(word2idx)
encoder = EncoderCNN(embed_size=512).to(device)
decoder = DecoderRNN(
    embed_size=512,
    hidden_size=512,
    vocab_size=vocab_size
).to(device)
encoder.load_state_dict(
    torch.load("/content/drive/MyDrive/image_captioning/encoder.pth", map_location=device)
)

decoder.load_state_dict(
    torch.load("/content/drive/MyDrive/image_captioning/decoder.pth", map_location=device)
)

encoder.eval()
decoder.eval()

print("Models loaded successfully")


In [ ]:
images, captions, lengths = next(iter(loader))
images = images.to(device)

with torch.no_grad():
    features = encoder(images)

print("Feature shape:", features.shape)
